In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace = True)

In [7]:
df.shape

(49582, 2)

## Pre-Processing

### 1. Converting to lowercase

In [8]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [9]:
import re

# sample_text = "abc is the word, abc"

# new_text = re.sub("abc", "xyz", sample_text)

In [10]:
def remove_urls(text):
    text = re.sub(r"http\S+", "", text) #(pattern, replace, string)  eg- https://www.google.com
    return text
    
df["review"] = df["review"].apply(remove_urls)

### 3. Removing Punctuations

In [11]:
def remove_punctuations(text):
    text =  re.sub(r"[^A-Za-z0-9\s]", "", text) #A-Z a-z 0-9 \s
    return text
    
df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML

In [12]:
def remove_html(text):
    text =  re.sub(r"<.*?>", "", text)
    return text
    
df["review"] = df["review"].apply(remove_html)

### Removing the stop word

In [13]:
!pip install nltk

In [14]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Subhajit
[nltk_data]     Jana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Subhajit
[nltk_data]     Jana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Subhajit
[nltk_data]     Jana\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [16]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [17]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### Steemming

In [18]:
# running ->run
# played -> play
#PoterStemming

from nltk.stem import PorterStemmer

In [19]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [20]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [22]:
y = df["sentiment"] 

In [23]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8.Vectorizarion

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [25]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>

### Dataset & DataLoaders

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [37]:
X_train.shape

(39665, 5000)

In [38]:
X_test.shape

(9917, 5000)

In [39]:
X_train =  X_train.toarray()
X_test = X_test.toarray()

In [40]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [41]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [42]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

### Build Our RNN

In [44]:
import torch.nn as nn
import torch.optim as optim

In [45]:
class RNN(nn.Module):
    def  __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        #optional => shape (num of layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden_size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [46]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training RNN

In [49]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb, in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)  #add singleton direction
        
        outputs = model(Xb)  # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,)

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.2871950566768646
epoch = 2/10 and loss = 0.24496281147003174
epoch = 3/10 and loss = 0.1642528623342514
epoch = 4/10 and loss = 0.1864665299654007
epoch = 5/10 and loss = 0.4134346842765808
epoch = 6/10 and loss = 0.23671701550483704
epoch = 7/10 and loss = 0.2432328313589096
epoch = 8/10 and loss = 0.19786903262138367
epoch = 9/10 and loss = 0.3062620460987091
epoch = 10/10 and loss = 0.29736000299453735


In [52]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb, in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"Accuracy = {correct_vals/tot_vals*100}")

Accuracy = 85.32822426136937
